In [1]:
import random
random.seed(42)

import torch

import numpy as np
import pyomo.environ as pyo
import pandas as pd

from pyomo.environ import *
from utils import generate_simple_synthetic_data, const_embed
from trainer import Trainer

np.random.seed(43)

In [2]:
!pip install gurobipy


In [3]:
import gurobipy

In [4]:
import os
os.environ["PATH"]

'c:\\Users\\DanielLosada\\anaconda3\\envs\\TFM;C:\\Users\\DanielLosada\\anaconda3\\envs\\TFM;C:\\Users\\DanielLosada\\anaconda3\\envs\\TFM\\Library\\mingw-w64\\bin;C:\\Users\\DanielLosada\\anaconda3\\envs\\TFM\\Library\\usr\\bin;C:\\Users\\DanielLosada\\anaconda3\\envs\\TFM\\Library\\bin;C:\\Users\\DanielLosada\\anaconda3\\envs\\TFM\\Scripts;C:\\Users\\DanielLosada\\anaconda3\\envs\\TFM\\bin;C:\\Users\\DanielLosada\\anaconda3\\condabin;C:\\WINDOWS\\system32;C:\\WINDOWS;C:\\WINDOWS\\System32\\Wbem;C:\\WINDOWS\\System32\\WindowsPowerShell\\v1.0;C:\\WINDOWS\\System32\\OpenSSH;C:\\Program Files\\Git\\cmd;C:\\Program Files\\Docker\\Docker\\resources\\bin;C:\\Program Files\\dotnet;C:\\glpk-4.65\\w64;C:\\Program Files\\MATLAB\\R2024b\\bin;C:\\Program Files\\Graphviz\\bin;C:\\Users\\DanielLosada\\anaconda3;C:\\Users\\DanielLosada\\anaconda3\\Library\\mingw-w64\\bin;C:\\Users\\DanielLosada\\anaconda3\\Library\\usr\\bin;C:\\Users\\DanielLosada\\anaconda3\\Library\\bin;C:\\Users\\DanielLosada\\an

In [5]:

X, y, true_mean, true_std = generate_simple_synthetic_data(n_samples=10)

print(X.head())
print(y.head())


         x1        x2     price
0  0.374540  0.020584  6.506676
1  0.950714  0.969910  2.255445
2  0.731994  0.832443  3.629302
3  0.598658  0.212339  4.297257
4  0.156019  0.181825  5.104630
0    0.064944
1    3.683185
2    2.038461
3    1.054569
4    0.232799
Name: y, dtype: float64


In [6]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

'''model_config = {
    'hidden_layers': 1,
    'hidden_size': 5,
    'drop': 0.2
    }

training_config = {
    'lr': 0.01,
    'wd': 0.0001,
    'batchsize': 64,
    'iters': 500,
    'patience': 20
    }'''

config = {
    # Model config
    'hidden_layers': 1,
    'hidden_size': 5,
    'drop': 0.2,

    # Training config
    'lr': 0.01,
    'wd': 0.0001,
    'batchsize': 64,
    'iters': 500,
    'patience': 20
}

trainer = Trainer(X, y, config, device=device)
trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)

trainer.train()

trainer.test()

X_train shape:  (6, 3)
X_val shape:  (2, 3)
X_test shape:  (2, 3)
Validation loss improved from inf to 0.6711 in epoch 1. Saving model...
Epoch 1/500 - Train Loss: 5.1256, Val Loss: 0.6711
Validation loss improved from 0.6711 to 0.5078 in epoch 2. Saving model...
Validation loss improved from 0.5078 to 0.3880 in epoch 3. Saving model...
Validation loss improved from 0.3880 to 0.2992 in epoch 4. Saving model...
Validation loss improved from 0.2992 to 0.2253 in epoch 5. Saving model...
Validation loss improved from 0.2253 to 0.1731 in epoch 6. Saving model...
Validation loss improved from 0.1731 to 0.1385 in epoch 7. Saving model...
Validation loss improved from 0.1385 to 0.1177 in epoch 8. Saving model...
Validation loss improved from 0.1177 to 0.1078 in epoch 9. Saving model...
Validation loss improved from 0.1078 to 0.1068 in epoch 10. Saving model...
Epoch 11/500 - Train Loss: 1.4115, Val Loss: 0.1115
Epoch 21/500 - Train Loss: 0.6891, Val Loss: 0.3125
Early stopping triggered after 

In [7]:
trainer.model

distModel(
  (input_layer): Linear(in_features=3, out_features=5, bias=True)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=5, out_features=5, bias=True)
  )
  (output_mean_layer): Linear(in_features=5, out_features=1, bias=True)
  (output_sd_layer): Linear(in_features=5, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)

In [8]:
cons = trainer.constraint_build()
cons.to_csv('constraints.csv')

In [9]:
cons = pd.read_csv('constraints.csv', index_col=0)
cons

,intercept,layer,node,node_0,node_1,node_2,node_3,node_4
0,-0.086398,0,0,0.258005,-0.227253,-0.067179,NaN,NaN
1,0.172099,0,1,0.295980,-0.622009,0.436071,NaN,NaN
2,-0.042946,0,2,-0.189990,0.232499,0.110008,NaN,NaN
3,0.417271,0,3,-0.001572,0.248744,0.034618,NaN,NaN
4,0.091117,0,4,0.207501,-0.151761,-0.108841,NaN,NaN
0,0.330658,1,0,-0.077806,-0.185207,0.018999,-0.092996,-0.047817
1,-0.139369,1,1,0.001868,0.168291,0.144993,-0.338088,0.178847
2,0.228436,1,2,0.212706,0.512586,0.327687,-0.312229,-0.337787
3,-0.033616,1,3,-0.159818,0.476937,0.012136,0.286325,-0.136516
4,-0.102237,1,4,0.396933,-0.091936,0.279095,-0.050825,-0.277473


In [10]:
n_sc = 3
contextual_var = trainer.X_test

In [15]:
model = pyo.ConcreteModel()
    
model.N = pyo.RangeSet(0,0)
model.W = pyo.RangeSet(1,n_sc) # scenarios

pi = [1/n_sc for i in range(n_sc)]

model.x = pyo.Var(contextual_var.columns.sort_values(), model.N)

model.y = pyo.Var(pyo.Any, dense=False, domain=pyo.Reals)

model.rent = pyo.Var(model.W, within=pyo.Reals) # rentabilidad per scenario

#definition of the objective function
def obj_expression(model):
    return sum(pi[w-1] * model.rent[w] for w in model.W)
model.OBJ = pyo.Objective(rule=obj_expression, sense=pyo.maximize)

#definition of the constraints

def rent_expression(model, w):
    return model.rent[w] == sum(model.y['demand', n, w]*model.x['price',n] for n in model.N)
model.rent_expression = pyo.Constraint(model.W, rule=rent_expression)

In [16]:
trainer.y_mean

array([1.19991147])

In [17]:
const_embed(model=trainer, opt_model=model, constaints=cons, outcome='demand', n_scenarios=n_sc)

In [20]:
opt = pyo.SolverFactory('gurobi_direct')
opt.options['threads'] = 8
opt.options['NonConvex'] = 2
results = opt.solve(model, tee=True)

Set parameter OutputFlag to value 1
Set parameter Threads to value 8
Set parameter NonConvex to value 2
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))



CPU model: AMD Ryzen 5 5625U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 8 threads

Non-default parameters:
NonConvex  2
Threads  8

Optimize a model with 35 rows, 31 columns and 151 nonzeros
Model fingerprint: 0x7fb31285
Model has 3 quadratic constraints
Variable types: 21 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [2e-03, 1e+03]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [3e-01, 3e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-02, 1e+03]
Presolve removed 2 rows and 5 columns
Presolve time: 0.00s
Presolved: 40 rows, 31 columns, 161 nonzeros
Presolved model has 3 bilinear constraint(s)
         in product terms.
         Presolve was not able to compute smaller bounds for these variables.
         Consider bounding these variables or reformulating the model.


Solving non-convex MIQCP

Variable types: 21 continuous, 1

In [19]:
for c in model.component_objects(Constraint, active=True):
    print(f"Constraint component: {c.name}")
    for index in c:
        print(f"  {index}: {c[index].expr}")


Constraint component: rent_expression
  1: rent[1]  ==  y[demand,0,1]*x[price,0]
  2: rent[2]  ==  y[demand,0,2]*x[price,0]
  3: rent[3]  ==  y[demand,0,3]*x[price,0]
Constraint component: constraint_1_layer0_node0_pred0demand
  None: (x[x1,0] - 0.5198339003650098)/0.30336634092621684*0.2580053 + (x[x2,0] - 0.3911390889126027)/0.22580128160063653*-0.22725254 + (x[price,0] - 3.8124026640068727)/1.414158920256157*-0.06717876 - 0.08639777  <=  v[demand,0,0,0]
Constraint component: constraint_2_layer0_node0_pred0demand
  None: v[demand,0,0,0]  <=  1000.0*v_ind[demand,0,0,0]
Constraint component: constraint_3_layer0_node0_pred0demand
  None: v[demand,0,0,0]  <=  (x[x1,0] - 0.5198339003650098)/0.30336634092621684*0.2580053 + (x[x2,0] - 0.3911390889126027)/0.22580128160063653*-0.22725254 + (x[price,0] - 3.8124026640068727)/1.414158920256157*-0.06717876 - 0.08639777 + 1000.0*(1 - v_ind[demand,0,0,0])
Constraint component: constraint_1_layer0_node1_pred0demand
  None: (x[x1,0] - 0.5198339003650